In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from stoneforge.data_management.preprocessing import DataLoader, DataManager

# Manual Access:
las2 = DataLoader(r"https://raw.githubusercontent.com/giecaruff/datasets/refs/heads/main/wells/las2/npra/IK1.las", filetype='las2')
print('header itens:',las2.data_obj.header.keys())
las2.data_obj.header['well']

header itens: dict_keys(['version', 'well', 'curve', 'parameter', 'other'])


,mnemonic,unit,value,description
0,STRT,F,81.0000,START DEPTH
1,STOP,F,15400.0000,STOP DEPTH
2,STEP,F,0.5000,STEP VALUE
3,NULL,,-999.2500,NULL VALUE
4,COMP,,USGS/NPR HUSKY OIL OPERAT,COMPANY
5,WELL,,IKPIKPUK TEST WELL #1,WELL
6,FLD,,WILDCAT,FIELD
7,LOC,,25 13N 10W,LOCATION
8,CNTY,,NORTH SLPOE,COUNTY
9,STAT,,ALASKA,STATE


In [3]:
# Example (Manual Access): Accessing data as DataFrame
data_las2, units_las2 = las2.dataframe(las2.data_obj.data)
data_las2 = data_las2.replace(-999.0, np.nan)
data_las2

,DEPT,SP,ILD,ILM,LL8,GR,CALI,RHOB,DRHO,NPHI,DT
0,81.0,NaN,NaN,NaN,NaN,79.7502,NaN,NaN,NaN,NaN,NaN
1,81.5,NaN,NaN,NaN,NaN,79.9790,NaN,NaN,NaN,NaN,NaN
2,82.0,NaN,NaN,NaN,NaN,79.8643,NaN,NaN,NaN,NaN,NaN
3,82.5,NaN,NaN,NaN,NaN,79.9446,NaN,NaN,NaN,NaN,NaN
4,83.0,NaN,NaN,NaN,NaN,80.1459,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
30796,15479.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30797,15479.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30798,15480.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30799,15480.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Adding facies
IK1 = DataManager(las2, depth="DEPT")

IK1.add_facie(name="LEDGE_SANDSTONE", top=10619, bottom=10842)

# View Facies LEDGE_SANDSTONE interval
IK1.LEDGE_SANDSTONE

,DEPT,SP,ILD,ILM,LL8,GR,CALI,RHOB,DRHO,NPHI,DT
21076,10619.0,-92.3719,5.6534,7.5047,10.3189,32.1681,9.5756,2.4324,0.0280,18.5964,74.0643
21077,10619.5,-92.6030,5.4512,7.1849,9.8774,29.9718,9.5682,2.4169,0.0243,18.5066,73.7811
21078,10620.0,-92.8108,5.2563,6.9832,9.3140,28.8898,9.5609,2.4278,0.0214,18.2585,73.3278
21079,10620.5,-93.0186,5.0684,6.7872,8.3432,28.8571,9.5536,2.4187,0.0232,18.0104,72.9256
21080,10621.0,-93.2263,4.9828,6.7115,7.1612,29.6683,9.5463,2.3921,0.0221,17.8458,73.9007
...,...,...,...,...,...,...,...,...,...,...,...
21518,10840.0,-84.3232,15.8031,24.9516,38.3127,38.1211,9.6802,2.5684,0.0246,9.2279,65.8097
21519,10840.5,-83.6495,18.1204,25.9377,29.7650,36.1407,9.6372,2.5561,0.0323,8.8146,63.3392
21520,10841.0,-82.9757,23.8513,37.9959,22.8938,36.0931,9.5941,2.5345,0.0364,8.6617,63.0884
21521,10841.5,-82.3019,24.9165,54.6157,25.3613,37.6945,9.5790,2.5176,0.0275,8.5357,63.5907


In [ ]:
class MineralDatabase:
    """Handles loading and parsing of petrophysical mineral endpoint properties

    from various file formats (CSV, JSON, or Dictionary).
    """

    def __init__(self, data_source=None):
        self.matrix = {}
        if data_source:
            if isinstance(data_source, dict):
                self.load_from_dict(data_source)
            elif isinstance(data_source, str):
                if data_source.endswith(".csv"):
                    self.load_from_csv(data_source)
                elif data_source.endswith(".json"):
                    self.load_from_json(data_source)

    def load_from_dict(self, d: dict):
        """Loads database from a standard python dictionary."""
        self.matrix = {
            str(min_k).upper(): {str(log_k).upper(): float(val) for log_k, val in log_v.items()}
            for min_k, log_v in d.items()
        }

    def load_from_json(self, filepath: str):
        """Loads database from a JSON file."""
        with open(filepath, "r") as f:
            raw_data = json.load(f)
        self.load_from_dict(raw_data)

    def load_from_csv(self, filepath: str):
        """Loads database from a standard CSV file."""
        df = pd.read_csv(filepath)
        # Ensure the first column contains the mineral/element identifier string
        first_col = df.columns[0]
        df[first_col] = df[first_col].astype(str).str.upper()
        df = df.set_index(first_col)

        # Convert dataframe directly to the nested dictionary configuration format
        raw_data = df.to_dict(orient="index")
        self.load_from_dict(raw_data)

    def get_matrix(self) -> dict:
        return self.matrix


class ElanInversion:
    """Highly optimized, loop-free Multimineral Linear Inversion Engine."""

    def __init__(self, dataframe: pd.DataFrame):
        self.df = dataframe.copy()
        self.df.columns = [str(col).upper() for col in self.df.columns]
        self.num_samples = len(self.df)

        # Inject the volume constraint parameter (Sum of Volumes = 1)
        self.df["ONES"] = np.ones(self.num_samples)

    def solve(
        self,
        db: MineralDatabase,
        active_logs: list,
        active_minerals: list,
        method: str = "least_squares",
        reg: float = 0.0,
        enforce_positive: bool = True,
    ) -> pd.DataFrame:
        """Dynamically builds matrix mappings based on selected logs/minerals and solves."""
        active_logs = [str(log).upper() for log in active_logs]
        active_minerals = [str(m).upper() for m in active_minerals]
        mineral_matrix = db.get_matrix()

        # Build local copies for constraint logic
        solver_logs = list(active_logs)
        if "ONES" not in solver_logs:
            solver_logs.append("ONES")

        # 1. Dynamically build Response Matrix (A) based strictly on active selections
        # Shape: (M logs, N minerals)
        try:
            A = []
            for log in solver_logs:
                row = []
                for mineral in active_minerals:
                    # Default to 1.0 if checking ONES constraint, else pull from DB
                    val = 1.0 if log == "ONES" else mineral_matrix[mineral][log]
                    row.append(val)
                A.append(row)
            A = np.array(A, dtype=float)
        except KeyError as e:
            raise KeyError(
                f"Requested pair not found in database. Check logs/mineral definitions: {e}"
            )

        # 2. Extract Data Matrix (B) from DataFrame -> Shape: (M logs, Depth Samples)
        B = self.df[solver_logs].to_numpy().T

        # 3. Regularization term
        identity_reg = np.eye(A.shape[1] if method != "exact" else A.shape[0]) * reg

        # 4. Solvers
        if method == "exact":
            A_inv = np.linalg.inv(A + identity_reg)
            X = np.dot(A_inv, B)
        elif method == "least_squares":
            ATA = np.dot(A.T, A)
            A_inv = np.linalg.inv(ATA + identity_reg)
            X = np.dot(np.dot(A_inv, A.T), B)
        elif method == "underdetermined":
            AAT = np.dot(A, A.T)
            A_inv = np.linalg.inv(AAT + identity_reg)
            X = np.dot(np.dot(A.T, A_inv), B)
        else:
            raise ValueError(f"Unknown method: {method}")

        X = X.T  # Transpose to shape: (Depth Samples, N minerals)

        # 5. Volumetric Adjustments (Non-negative & Unity Constraint closure)
        if enforce_positive:
            X = np.clip(X, a_min=0, a_max=None)
            row_sums = X.sum(axis=1, keepdims=True)
            row_sums[row_sums == 0] = 1.0
            X = X / row_sums

        return pd.DataFrame(X, columns=active_minerals, index=self.df.index)

In [23]:
MINERAL_DATABASE = {
    "QUARTZ": {"GR": 20.0, "DT": 55.5, "RHOB": 2.650, "NPHI": 0.00, "ONES": 1.0},
    "CALCITE": {"GR": 11.0, "DT": 47.8, "RHOB": 2.710, "NPHI": 0.00, "ONES": 1.0},
    "SHALE": {"GR": 111.0, "DT": 100.0, "RHOB": 2.657, "NPHI": 48.1, "ONES": 1.0},
    "ARKOSE": {"GR": 160.0, "DT": 130.0, "RHOB": 2.560, "NPHI": 40.0, "ONES": 1.0},
    "FLUID": {"GR": 0.0001, "DT": 185.0, "RHOB": 1.100, "NPHI": 100.0, "ONES": 1.0}
}

IK1.LEDGE_SANDSTONE

,DEPT,SP,ILD,ILM,LL8,GR,CALI,RHOB,DRHO,NPHI,DT
21076,10619.0,-92.3719,5.6534,7.5047,10.3189,32.1681,9.5756,2.4324,0.0280,18.5964,74.0643
21077,10619.5,-92.6030,5.4512,7.1849,9.8774,29.9718,9.5682,2.4169,0.0243,18.5066,73.7811
21078,10620.0,-92.8108,5.2563,6.9832,9.3140,28.8898,9.5609,2.4278,0.0214,18.2585,73.3278
21079,10620.5,-93.0186,5.0684,6.7872,8.3432,28.8571,9.5536,2.4187,0.0232,18.0104,72.9256
21080,10621.0,-93.2263,4.9828,6.7115,7.1612,29.6683,9.5463,2.3921,0.0221,17.8458,73.9007
...,...,...,...,...,...,...,...,...,...,...,...
21518,10840.0,-84.3232,15.8031,24.9516,38.3127,38.1211,9.6802,2.5684,0.0246,9.2279,65.8097
21519,10840.5,-83.6495,18.1204,25.9377,29.7650,36.1407,9.6372,2.5561,0.0323,8.8146,63.3392
21520,10841.0,-82.9757,23.8513,37.9959,22.8938,36.0931,9.5941,2.5345,0.0364,8.6617,63.0884
21521,10841.5,-82.3019,24.9165,54.6157,25.3613,37.6945,9.5790,2.5176,0.0275,8.5357,63.5907


In [26]:
elan = ElanInversion(IK1.LEDGE_SANDSTONE)

solved_volumes = elan.solve(
    mineral_matrix=MINERAL_DATABASE,
    active_logs=["GR", "DT", "RHOB", "NPHI"],
    active_minerals=["QUARTZ", "SHALE", "FLUID"],  # Easily swap minerals out!
    method="least_squares",
    enforce_positive=True,
)

In [27]:
print(solved_volumes)

         QUARTZ     SHALE     FLUID
21076  0.714371  0.174979  0.110650
21077  0.723559  0.155408  0.121033
21078  0.730191  0.145531  0.124278
21079  0.732634  0.145258  0.122108
21080  0.738685  0.145343  0.115972
...         ...       ...       ...
21518  0.813744  0.186256  0.000000
21519  0.817429  0.180628  0.001943
21520  0.819026  0.180293  0.000681
21521  0.811342  0.188658  0.000000
21522  0.802608  0.197392  0.000000

[447 rows x 3 columns]
